# LSH Band Decomposition: Intuition and Parameter Tuning

## Learning Objectives
- Understand intuitively why splitting signatures into bands amplifies the similarity signal
- Interpret S-curves and their relationship to false positives and false negatives
- Tune the (b, r) parameters for a given similarity threshold and precision/recall target
- Diagnose common mistakes in LSH configuration

This notebook is a conceptual companion to `ml_002_03_lsh_algorithm.ipynb`, which contains the full implementation. Here we focus on intuition and parameter tuning.

## The Core Intuition: Why Band Decomposition Works

Consider documents A and B with Jaccard J=0.7 and a signature of length k=100.

**Naive approach**: declare a candidate pair if the fraction of matching signature rows exceeds a threshold.

Problem: the fraction of matching rows is approximately 0.7 with std approximately 1/(2*sqrt(100)) = 0.05. The decision boundary is blurry — there is substantial probability of including pairs at J=0.5 and missing pairs at J=0.9.

**Band decomposition**: divide the 100 rows into b=20 bands of r=5 rows each. Declare a candidate if *any* band has *all* r rows matching.

- Within a band, P[all r rows agree] = J^r = 0.7^5 approx 0.168
- Across b bands, P[at least one band agrees] = 1 - (1 - 0.168)^20 approx 0.975
- For J=0.4: 0.4^5 approx 0.010, then 1-(0.99)^20 approx 0.182

The band structure creates a *step function*: pairs above the threshold are almost always found; pairs below are almost always skipped.

**Analogy**: imagine a jury of 20 subcommittees, each with 5 members. A verdict requires unanimous agreement in *at least one* subcommittee. A guilty (similar) pair will convince at least one committee easily; an innocent (dissimilar) pair rarely achieves unanimity anywhere.

The AND operation within a band sharpens the all-or-nothing signal; the OR across bands recovers sensitivity.

## S-Curve: Result and Intuition

The full derivation is in `ml_002_03_lsh_algorithm.ipynb`. The key result:

$$P[\text{candidate} \mid J] = 1 - (1 - J^r)^b$$

Two design choices create the S-shape:

1. **AND within band** (J^r): the probability of a band firing is the product of r independent agreement probabilities. Products of numbers less than 1 fall fast — this suppresses low-J pairs steeply.
2. **OR across bands** (1-(1-p)^b): the probability of missing a fire decreases exponentially in b. This boosts high-J pairs toward probability 1.

The approximate threshold (steepest point) is $t^* \approx (1/b)^{1/r}$, which is the J value where b * J^r = 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def s_curve(J, b, r):
    # P[at least one band fully agrees]
    return 1 - (1 - J**r)**b

j_vals = np.linspace(0, 1, 500)

configs = [
    (20,  5, 'b=20, r=5  (t*~0.55)'),
    (10, 10, 'b=10, r=10 (t*~0.79)'),
    (50,  2, 'b=50, r=2  (t*~0.98)'),
    ( 5, 20, 'b=5,  r=20 (t*~0.87)'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# left: all curves together
ax = axes[0]
for b, r, label in configs:
    ax.plot(j_vals, s_curve(j_vals, b, r), lw=2, label=label)
    t_star = (1/b)**(1/r)
    ax.axvline(t_star, color='gray', linestyle=':', lw=0.8, alpha=0.6)
ax.axhline(0.5, color='black', linestyle='--', lw=0.8, label='P=0.5 (threshold line)')
ax.set_xlabel('Jaccard similarity J')
ax.set_ylabel('P[candidate pair]')
ax.set_title('S-curves for k=100, various (b, r)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# right: zoom on threshold region 0.3-0.9
ax = axes[1]
j_zoom = np.linspace(0.3, 0.9, 500)
for b, r, label in configs:
    ax.plot(j_zoom, s_curve(j_zoom, b, r), lw=2, label=label)
ax.axhline(0.5, color='black', linestyle='--', lw=0.8)
ax.set_xlabel('Jaccard similarity J')
ax.set_ylabel('P[candidate pair]')
ax.set_title('S-curves (zoomed, 0.3-0.9)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lsh_s_curves.png', dpi=100)
plt.show()
print('saved lsh_s_curves.png')

## Parameter Tuning Guide

Given a target threshold t and a signature length k, how do you choose b and r?

**Hard constraint**: b * r = k (the bands must partition the signature exactly).

**Objective**: pick (b, r) such that:
- The S-curve crosses 0.5 near t (correct threshold)
- The S-curve is as steep as possible (sharp decision boundary)
- FPR at some 'definitely-not-similar' J value is below a budget
- TPR at some 'definitely-similar' J value is above a budget

The S-curve slope at the threshold is maximized by making both b and r large, which requires large k.

In [ ]:
def suggest_params(threshold, k, max_fp_at=None, min_tp_at=None, top_n=5):
    # threshold : target similarity threshold (S-curve should cross 0.5 near here)
    # k         : signature length (b*r must equal k)
    # max_fp_at : J below threshold where FPR should be low (default: threshold-0.2)
    # min_tp_at : J above threshold where TPR should be high (default: threshold+0.2)
    if max_fp_at is None:
        max_fp_at = max(0.0, threshold - 0.2)
    if min_tp_at is None:
        min_tp_at = min(1.0, threshold + 0.2)

    candidates = []
    for b in range(1, k + 1):
        if k % b != 0:
            continue
        r = k // b
        p_at_t  = s_curve(threshold, b, r)
        fp      = s_curve(max_fp_at, b, r)
        tp      = s_curve(min_tp_at, b, r)
        t_error = abs(p_at_t - 0.5)
        candidates.append((b, r, p_at_t, fp, tp, t_error))

    # sort: closest P(t) to 0.5 first, then by tp-fp margin
    candidates.sort(key=lambda x: (x[5], -(x[4] - x[3])))

    print(f'Parameter suggestions for threshold={threshold}, k={k}')
    print(f'  FP reference J = {max_fp_at:.2f},  TP reference J = {min_tp_at:.2f}')
    hdr = f'  {"b":>4} {"r":>4}  {"P(t*)":>6}  {"FP":>6}  {"TP":>6}  {"TP-FP":>6}'
    print(hdr)
    for b, r, p_t, fp, tp, _ in candidates[:top_n]:
        print(f'  {b:4d} {r:4d}  {p_t:6.3f}  {fp:6.3f}  {tp:6.3f}  {tp-fp:6.3f}')
    return candidates[:top_n]

print('=== Target threshold=0.6, k=100 ===')
suggest_params(0.6, 100)
print()
print('=== Target threshold=0.8, k=100 ===')
suggest_params(0.8, 100)
print()
print('=== Target threshold=0.5, k=200 ===')
suggest_params(0.5, 200)

## False Positive and False Negative Analysis

Given the S-curve P(J) = 1 - (1 - J^r)^b and a threshold t:

- **False positive rate at J**: P_FP(J) = P(J) for J < t — the probability that a dissimilar pair is returned as a candidate
- **False negative rate at J**: P_FN(J) = 1 - P(J) for J >= t — the probability that a similar pair is *missed*

Note that FPR and FNR are *not* single numbers — they depend on the actual J of the pair. The S-curve is the complete picture.

A *sharp* S-curve (steep slope near t) minimizes the area under the FP/FN curves near the threshold. A *shifted* S-curve (threshold not at t) creates systematic bias.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# compare two configurations for threshold=0.6, k=100
threshold = 0.6
j_vals = np.linspace(0, 1, 500)

params_to_compare = [
    (20,  5, 'b=20, r=5  (t*~0.55, near 0.6)'),
    (10, 10, 'b=10, r=10 (t*~0.79, too high)'),
]
colors = ['steelblue', 'tomato']
ax_fp, ax_fn = axes

for (b, r, label), color in zip(params_to_compare, colors):
    fp_region = j_vals[j_vals < threshold]
    fp_vals   = s_curve(fp_region, b, r)
    fn_region = j_vals[j_vals >= threshold]
    fn_vals   = 1 - s_curve(fn_region, b, r)
    ax_fp.plot(fp_region, fp_vals, color=color, lw=2, label=label)
    ax_fn.plot(fn_region, fn_vals, color=color, lw=2, label=label)

ax_fp.axvline(threshold, color='black', linestyle='--', lw=1, label=f'threshold={threshold}')
ax_fn.axvline(threshold, color='black', linestyle='--', lw=1, label=f'threshold={threshold}')

ax_fp.set_xlabel('True Jaccard J  (J < threshold)')
ax_fp.set_ylabel('P[false positive | J]')
ax_fp.set_title('False Positive Rate by J')
ax_fp.legend(fontsize=8)
ax_fp.grid(True, alpha=0.3)

ax_fn.set_xlabel('True Jaccard J  (J >= threshold)')
ax_fn.set_ylabel('P[false negative | J]')
ax_fn.set_title('False Negative Rate by J')
ax_fn.legend(fontsize=8)
ax_fn.grid(True, alpha=0.3)

plt.suptitle('FP and FN profiles for threshold=0.6, k=100', fontsize=12)
plt.tight_layout()
plt.savefig('lsh_fp_fn_analysis.png', dpi=100)
plt.show()
print('saved lsh_fp_fn_analysis.png')
print()
for b, r, label in params_to_compare:
    fp_at_04 = s_curve(0.4, b, r)
    fn_at_08 = 1 - s_curve(0.8, b, r)
    print(f'  {label}')
    print(f'    FP at J=0.4: {fp_at_04:.4f}  |  FN at J=0.8: {fn_at_08:.4f}')

## Common Mistakes in LSH Configuration

### 1. Choosing r=1

With r=1, each band is a single row. The S-curve becomes:

$$P = 1 - (1 - J^1)^b = 1 - (1-J)^b$$

This is just a smoothed version of plain signature agreement — no amplification. You lose the AND step that creates the sharp threshold.

### 2. Choosing b=1

With b=1, there is only one band of r=k rows. The S-curve becomes:

$$P = J^k$$

This drops to 0 very fast — only pairs with J near 1 are ever candidates. Useful if you specifically want near-exact duplicates, but misses everything below J~0.9 for k>=20.

### 3. Making k too small before choosing b and r

If k=10, then b*r=10. The possible (b,r) pairs are severely limited: (1,10),(2,5),(5,2),(10,1). None of these produce a good S-curve for a 0.5-0.7 threshold. Fix: increase k first (k=100 or k=200).

### 4. Using the approximate threshold formula without verifying

The formula t* ~ (1/b)^(1/r) is a first-order approximation. Always evaluate the *actual* S-curve at your target J values using P = 1-(1-J^r)^b to check actual FPR and FNR before deploying.

### 5. Confusing the threshold for 'precision = recall = 1'

The S-curve never reaches 0 below the threshold or 1 above it (for finite b,r). There is always some FP and FN probability. The goal is to make both *acceptably small* for your application, not to eliminate them entirely.